# 4교시. 멀티모달·생성형 AI 기반 핵심 정보 추출

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/master/colab/04_genai_extraction.ipynb)

## 오늘 꼭 할 일

공개 영수증을 실제 문서 VLM으로 읽고 업무 JSON과 원본 근거를 확인합니다.

1. 제공 예제로 결과를 먼저 만듭니다.
2. 화면에서 이번 교시의 핵심 결과 한 가지를 확인합니다.
3. 시간이 남으면 다른 공개·비식별 자료로 반복하고 차이를 기록합니다.

**끝났다는 증거:** 화면의 `✅ 실습 완료`와
`course_outputs/paddleocr_vl_result.md` 파일

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

필수 실습에서는 제공 예제를 사용합니다. 다른 자료를 사용한 경우에는
화면에 표시된 파일명이 내가 선택한 파일과 같은지 먼저 확인합니다.
4교시는 제공 이미지를 실제 VLM으로 읽어야 완료됩니다. GPU가 없거나
모델 실행이 실패하면 준비 결과로 성공 처리하지 않습니다. T4 GPU를
선택하고 GPU 확인 셀부터 다시 실행합니다.


## 이 노트북에서 내가 하는 일

- **필수 실습:** 공개 영수증 한 장을 PaddleOCR-VL-1.6으로 직접 읽고 OCR+규칙 결과와 비교합니다.
- **내가 바꾸는 곳:** 이미지 출처와 Excel 저장 전 검토 결정 세 곳만 바꿉니다.
- **인터넷 자료로 다시 실험:** 다른 공개·비식별 이미지로 실제 VLM을 다시 실행해 잘 읽은 구조와 빠진 필드를 기록합니다.

먼저 제공 예제로 끝까지 실행해 `✅ 실습 완료`를 확인하세요. 그다음
[공개·비식별 실습 자료 찾기](https://github.com/leecks1119/document_ai_lecture/blob/master/docs/public_practice_sources.md)를 보고
입력 한 장만 바꾸어 다시 실행합니다. 2교시에서 만든 결과 파일은
3~7교시에 이어 쓸 수 있습니다. 매 교시 마지막의 **다른 자료 실험
기록**에서 잘된 점과 실패한 점을 남깁니다.

> `🟢 그대로 실행하는 셀`은 수정하지 않습니다. `🟠 내가 짧게 바꾸는
> 셀`만 필수이고, `🔵 원하면 바꾸는 셀`은 시간이 남을 때 합니다.
> 정답은 모두 공개되어 있으므로 정답을 먼저 복사하고 결과를 관찰해도 됩니다.

## 코드 셀을 읽는 방법

각 코드 셀의 맨 위에는 `코드 읽기` 주석이 있습니다.

1. `수정하지 않습니다`라고 적힌 셀은 설명을 읽고 그대로 실행합니다.
2. 주황색 필수 `TODO`만 채웁니다. 파란색 선택 `TODO`는 건너뛰어도 됩니다.
3. 실행 출력에서 `코드 읽는 법`과 `확인할 결과`를 다시 확인합니다.
4. `단계 실행 완료`가 나온 뒤 다음 코드 셀로 이동합니다.
5. 길고 어려운 준비 코드는 접혀 있습니다. 제목 왼쪽의 화살표를 눌러
   펼칠 수 있지만, 처음에는 펼치지 않아도 됩니다.

Python 문법 전체를 먼저 이해할 필요는 없습니다. 변수에 어떤 값이 들어가고,
실행 뒤 어떤 결과가 달라지는지를 중심으로 읽습니다.


In [ ]:
#@title 🟢 0. 실습 환경 준비 — 그대로 실행 { display-mode: "form" }
def _show_learning_message(markdown_text):
    try:
        from IPython.display import Markdown, display
        display(Markdown(markdown_text))
    except ImportError:
        print(markdown_text)


def show_lab_step(
    current,
    total,
    title,
    action,
    expected,
    code_help,
    edit_kind,
):
    cell_kind = {
        "required": "🟠 내가 짧게 바꾸는 셀",
        "optional": "🔵 원하면 바꾸는 셀",
        "none": "🟢 그대로 실행하는 셀",
    }[edit_kind]
    _show_learning_message(
        f"""---
### {cell_kind} · {current}/{total} · {title}

**지금 할 일:** {action}

**코드 읽는 법:** {code_help}

**이 단계에서 확인할 결과:** {expected}
"""
    )


def complete_lab_step(current, total, expected):
    next_action = (
        "결과를 확인한 뒤 다음 코드 셀을 실행하세요."
        if current < total
        else "마지막 실습 완료 문구와 산출물 파일을 확인하세요."
    )
    _show_learning_message(
        f"""> ✅ **{current}/{total} 단계 실행 완료**
>
> **결과 확인:** {expected}
>
> **다음 행동:** {next_action}
"""
    )

# ── 코드 읽기 ─────────────────────────────────────────────
# `OUTPUT_DIR`는 모델 원본 결과와 업무 JSON을 모으는 폴더이고 `load_course_assets()`는 공개 이미지를 받습니다.
# 수정하지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(1, 9, '공통 환경 준비', '모델 결과를 저장할 폴더와 실습 자료 다운로드 기능을 준비합니다.', 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.', '`OUTPUT_DIR`는 모델 원본 결과와 업무 JSON을 모으는 폴더이고 `load_course_assets()`는 공개 이미지를 받습니다. 수정하지 않습니다.', 'none')

import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
AUTOMATED_CHECK = os.getenv("COURSE_VALIDATE_EXAMPLE") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or AUTOMATED_CHECK:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 자료 선택에서 "
            "'제공 예제'를 고르거나 파일을 다시 선택하세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if AUTOMATED_CHECK:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())

COURSE_ASSET_BASE_URL = (
    "https://raw.githubusercontent.com/leecks1119/"
    "document_ai_lecture/master/"
)

def load_course_assets(*relative_paths):
    if AUTOMATED_CHECK:
        local_root = os.getenv("COURSE_LOCAL_ASSET_ROOT")
        if not local_root:
            raise RuntimeError(
                "자동 검증용 COURSE_LOCAL_ASSET_ROOT가 필요합니다."
            )
        root = Path(local_root)
        return {
            path: (root / path).read_bytes()
            for path in relative_paths
        }

    import requests

    loaded = {}
    missing = []
    for path in relative_paths:
        try:
            response = requests.get(
                COURSE_ASSET_BASE_URL + path,
                timeout=30,
            )
            response.raise_for_status()
            loaded[path] = response.content
        except requests.RequestException as exc:
            print(f"자동 다운로드 실패: {Path(path).name} · {exc}")
            missing.append(path)

    if missing:
        from google.colab import files

        expected = ", ".join(Path(path).name for path in missing)
        print("다음 파일을 저장소에서 내려받아 선택하세요:", expected)
        uploaded = files.upload()
        uploaded_by_name = {
            Path(name).name: content
            for name, content in uploaded.items()
        }
        for path in missing:
            filename = Path(path).name
            if filename not in uploaded_by_name:
                raise FileNotFoundError(
                    f"{filename}이 선택되지 않았습니다."
                )
            loaded[path] = uploaded_by_name[filename]

    return loaded

complete_lab_step(1, 9, 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.')


In [ ]:
#@title 🟢 규칙 추출 함수 준비 — 그대로 실행 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# `extract_receipt_from_text()`는 정규식으로 날짜·합계·품목을 찾고 각 값의 원문 근거까지 함께 반환합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(2, 9, '규칙 추출 함수 준비', '날짜·합계·품목·근거를 JSON으로 만드는 함수를 등록합니다.', '오류 없이 끝나면 추출 함수를 사용할 수 있습니다.', '`extract_receipt_from_text()`는 정규식으로 날짜·합계·품목을 찾고 각 값의 원문 근거까지 함께 반환합니다.', 'none')

import re

def to_int(value):
    return int(value.replace(",", ""))


def extract_receipt_from_text(text, result_source):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    date_match = re.search(r"\b(\d{4})[-./](\d{1,2})[-./](\d{1,2})\b", text)
    total_line = next(
        (
            line
            for line in lines
            if re.search(r"(?:합\s*계|결제\s*금액|총\s*액)", line)
        ),
        None,
    )
    total_candidates = (
        re.findall(r"(?<![\d,])\d[\d,]*(?![\d,])", total_line)
        if total_line
        else []
    )
    total_raw = total_candidates[-1] if total_candidates else None
    supply_match = re.search(
        r"(?:부가세\s*)?과세물품가액\s*[:：]?\s*([\d,]+)",
        text,
    )
    vat_match = re.search(
        r"^부가세(?!\s*과세물품가액)\s*[:：]?\s*([\d,]+)",
        text,
        re.MULTILINE,
    )
    item_pattern = re.compile(
        r"^(?P<name>.+?)\s+(?P<unit>[\d,]+)\s+"
        r"(?P<quantity>\d+)\s+(?P<line>[\d,]+)$"
    )
    markdown_item_pattern = re.compile(
        r"^\|\s*(?P<name>[^|]+?)\s*\|\s*(?P<quantity>\d+)\s*\|"
        r"\s*(?P<unit>[\d,]+)원\s*\|\s*(?P<line>[\d,]+)원\s*\|$"
    )
    items = []
    item_evidence = []
    for line_number, line in enumerate(lines, start=1):
        match = item_pattern.search(line)
        if not match:
            match = markdown_item_pattern.search(line)
        if match:
            item = {
                "name": match.group("name"),
                "quantity": int(match.group("quantity")),
                "unit_price": to_int(match.group("unit")),
                "line_total": to_int(match.group("line")),
            }
            items.append(item)
            item_evidence.append({"line": line_number, "raw_value": line})

    date_value = (
        f"{int(date_match.group(1)):04d}-{int(date_match.group(2)):02d}-"
        f"{int(date_match.group(3)):02d}"
        if date_match else None
    )
    total_value = to_int(total_raw) if total_raw else None
    supply_value = to_int(supply_match.group(1)) if supply_match else None
    vat_value = to_int(vat_match.group(1)) if vat_match else None
    return {
        "document_type": "receipt",
        "store_name": lines[0] if lines else None,
        "date": date_value,
        "total_amount": total_value,
        "items": items,
        "adjustments": {"discount": 0, "tax": 0, "service": 0, "rounding": 0},
        "tax_breakdown": {
            "mode": "included_in_item_prices",
            "supply_amount": supply_value,
            "vat": vat_value,
            "payable_total": total_value,
        } if supply_value is not None and vat_value is not None else None,
        "raw_values": {
            "store_name": lines[0] if lines else None,
            "date": date_match.group(0) if date_match else None,
            "total_amount": total_raw,
        },
        "cleaned_values": {
            "store_name": lines[0] if lines else None,
            "date": date_value,
            "total_amount": total_value,
        },
        "evidence": {
            "store_name": {"line": 1, "raw_value": lines[0] if lines else None},
            "date": {"raw_value": date_match.group(0) if date_match else None},
            "total_amount": {"raw_value": total_line},
            "items": item_evidence,
        },
        "result_source": result_source,
    }

complete_lab_step(2, 9, '오류 없이 끝나면 추출 함수를 사용할 수 있습니다.')


## 이번 실습은 실제 문서 VLM을 실행합니다

사용하는 전체 파이프라인은 **PaddleOCR-VL-1.6**이고, 그 안에서
이미지를 언어와 함께 처리하는 모델은
**PaddleOCR-VL-1.6-0.9B**입니다.

```text
현재 이미지 픽셀
  → 문서 레이아웃 영역 탐지
  → 영역 자르기와 읽기 순서 결정
  → PaddleOCR-VL-1.6-0.9B가 각 영역 인식
  → Markdown·구조 JSON 조립
  → 업무용 영수증 JSON 변환
```

PaddleOCR-VL은 문서를 Markdown과 구조 JSON으로 복원하는 문서 전용
VLM입니다. 상호명·합계 같은 회사 업무 필드로 옮기는 마지막 단계는
아래의 투명한 Python 규칙이 담당합니다. 이 둘을 한 모델의 능력처럼
섞어 부르지 않습니다.

이 실습은 API 키와 건당 호출 비용은 없지만 Colab GPU와 최초 모델
다운로드가 필요합니다. GPU가 없거나 실제 추론이 실패하면 성공한
예제 결과로 몰래 바꾸지 않고 그 셀에서 멈춥니다.


In [ ]:
#@title 🔵 실습 자료 고르기 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# `실습_자료`에서 제공 예제·내 파일·인터넷 이미지 주소를 고릅니다. `VLM_INPUT_PATH`가 실제 모델에 전달할 현재 이미지입니다.
# ──────────────────────────────────────────────────────────
show_lab_step(3, 9, 'VLM 입력 이미지 준비', '제공 예제·내 파일·인터넷 주소 중 이미지 한 장을 선택합니다.', '선택한 자료·파일명·이미지 크기와 원본 화면을 확인합니다.', '`실습_자료`에서 제공 예제·내 파일·인터넷 이미지 주소를 고릅니다. `VLM_INPUT_PATH`가 실제 모델에 전달할 현재 이미지입니다.', 'optional')

# INPUT_FORM_CELL
import io
import requests
from PIL import Image
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

SAMPLE_IMAGE_PATH = (
    "sample_docs/public_receipts/korea/"
    "taebaek_restaurant_2025_redacted.png"
)
provided_bytes = load_course_assets(SAMPLE_IMAGE_PATH)[SAMPLE_IMAGE_PATH]

# TODO(선택): 제공 예제를 끝낸 뒤 이미지 한 장만 바꾸어 다시 실행하세요.
실습_자료 = "제공 예제" #@param ["제공 예제", "내 컴퓨터에서 업로드", "인터넷 이미지 URL"]
인터넷_이미지_URL = "" #@param {type:"string"}
if AUTOMATED_CHECK:
    실습_자료 = "제공 예제"

input_bytes = provided_bytes
INPUT_FILE_NAME = "taebaek_restaurant_2025_redacted.png"
if 실습_자료 == "내 컴퓨터에서 업로드":
    from google.colab import files
    print(
        "JPG·JPEG·PNG·WEBP 한 장을 선택하세요. "
        "개인·회사 문서는 식별정보를 먼저 가립니다."
    )
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("문서 이미지 한 장만 선택하세요.")
    INPUT_FILE_NAME, input_bytes = next(iter(uploaded.items()))
elif 실습_자료 == "인터넷 이미지 URL":
    if not 인터넷_이미지_URL.strip():
        raise ValueError("인터넷_이미지_URL에 이미지 주소를 붙여 넣으세요.")
    response = requests.get(인터넷_이미지_URL.strip(), timeout=30)
    response.raise_for_status()
    input_bytes = response.content
    from urllib.parse import urlparse
    INPUT_FILE_NAME = (
        Path(urlparse(인터넷_이미지_URL).path).name
        or "internet_document.png"
    )

if len(input_bytes) > 5 * 1024 * 1024:
    raise ValueError("수업에서는 5MB 이하 이미지 한 장만 사용합니다.")
try:
    vlm_input_image = Image.open(io.BytesIO(input_bytes)).convert("RGB")
except Exception as exc:
    raise ValueError(
        "웹페이지가 아니라 JPG·JPEG·PNG·WEBP 이미지 자체가 필요합니다."
    ) from exc

suffix = Path(INPUT_FILE_NAME).suffix.lower()
if suffix not in {".jpg", ".jpeg", ".png", ".webp"}:
    suffix = ".png"
VLM_INPUT_PATH = OUTPUT_DIR / f"vlm_input{suffix}"
vlm_input_image.save(VLM_INPUT_PATH)

previous_path = OUTPUT_DIR / "clean_receipt.json"
if previous_path.exists():
    clean_result = json.loads(previous_path.read_text(encoding="utf-8"))
    ocr_baseline_text = "\n".join(clean_result["cleaned_lines"])
    OCR_BASELINE_SOURCE = "3교시 정리 결과"
elif 실습_자료 == "제공 예제":
    ocr_baseline_text = '이태리집\n거래일시 2025-10-04 12:33:37\n페퍼로니 앤 치즈 29,000 1 29,000\n토마토 파스타 14,000 1 14,000\n수제 돈가스 13,000 1 13,000\n새우 칠리치 필라 14,000 1 14,000\n콜라 2,000 3 6,000\n합계 금액 76,000\n부가세 과세물품가액 69,094\n부가세 6,906\n'
    OCR_BASELINE_SOURCE = "제공 예제의 사람이 확인한 OCR 원문"
else:
    ocr_baseline_text = ""
    OCR_BASELINE_SOURCE = "비교할 OCR 원문 없음"

display(vlm_input_image)
print("선택한 자료:", 실습_자료)
print("실제 VLM 입력 파일:", INPUT_FILE_NAME)
print("이미지 크기:", vlm_input_image.size)
print("OCR 비교 자료:", OCR_BASELINE_SOURCE)

complete_lab_step(3, 9, '선택한 자료·파일명·이미지 크기와 원본 화면을 확인합니다.')


## Colab에서 GPU를 켭니다

메뉴에서 `런타임 → 런타임 유형 변경 → T4 GPU`를 선택한 뒤 아래 셀을
실행합니다. GPU 이름이 보이지 않으면 모델 셀로 넘어가지 않습니다.

설치하는 `paddleocr[doc-parser]`는 문서 파싱 파이프라인이고,
`transformers`는 이 실습에서 실제 VLM 추론을 수행하는 엔진입니다.


In [ ]:
#@title 🟢 GPU와 VLM 실행환경 확인 — 그대로 실행 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# `torch.cuda.is_available()`로 GPU를 확인한 뒤 `paddleocr[doc-parser]`와 `transformers`를
# 설치합니다. GPU가 없으면 안내대로 런타임을 바꿉니다.
# ──────────────────────────────────────────────────────────
show_lab_step(4, 9, 'GPU와 VLM 실행환경 확인', 'Colab GPU를 확인하고 PaddleOCR-VL 실행 패키지를 설치합니다.', 'GPU 이름·파이프라인·VLM 모델명이 표시되어야 합니다.', '`torch.cuda.is_available()`로 GPU를 확인한 뒤 `paddleocr[doc-parser]`와 `transformers`를 설치합니다. GPU가 없으면 안내대로 런타임을 바꿉니다.', 'none')

import subprocess

VLM_PIPELINE_NAME = "PaddleOCR-VL-1.6"
VLM_MODEL_NAME = "PaddleOCR-VL-1.6-0.9B"
VLM_ENGINE = "transformers"

if AUTOMATED_CHECK:
    VLM_RUNTIME_READY = False
    print("저장소 자동검사: 거대 모델 다운로드만 생략합니다.")
else:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError(
            "GPU가 없습니다. 런타임 → 런타임 유형 변경 → "
            "T4 GPU를 선택하고 이 셀부터 다시 실행하세요."
        )
    print("GPU:", torch.cuda.get_device_name(0))
    print("실행환경을 설치합니다. 최초 한 번은 몇 분 걸릴 수 있습니다.")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "paddleocr[doc-parser]==3.7.0",
        "transformers>=5.8,<6",
    ])
    VLM_RUNTIME_READY = True

print("전체 파이프라인:", VLM_PIPELINE_NAME)
print("실제 VLM 모델:", VLM_MODEL_NAME)
print("추론 엔진:", VLM_ENGINE)

complete_lab_step(4, 9, 'GPU 이름·파이프라인·VLM 모델명이 표시되어야 합니다.')


## 현재 이미지를 실제 VLM으로 읽습니다

아래 셀은 `VLM_INPUT_PATH`의 픽셀을 모델에 직접 전달합니다.
처음 실행할 때 공식 모델 파일을 내려받기 때문에 시간이 더 걸립니다.
결과가 나오면 Markdown과 원시 JSON을 모두 저장합니다.

오류가 나면 메시지를 그대로 확인합니다. 이 셀에는 준비된 VLM 결과로
대신하는 코드가 없습니다.


In [ ]:
#@title 🟢 PaddleOCR-VL 실제 실행 — 그대로 실행 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# `PaddleOCRVL`은 `pipeline_version='v1.6'`과 `engine='transformers'`로 현재 이미지에 실제
# 추론합니다. 실패 시 예제 결과로 바꾸지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(5, 9, 'PaddleOCR-VL 실제 실행', '현재 이미지 픽셀을 PaddleOCR-VL-1.6에 전달해 Markdown과 JSON을 만듭니다.', '`모델 실행 완료: True`와 실제 Markdown 일부를 확인합니다.', "`PaddleOCRVL`은 `pipeline_version='v1.6'`과 `engine='transformers'`로 현재 이미지에 실제 추론합니다. 실패 시 예제 결과로 바꾸지 않습니다.", 'none')

def json_safe(value):
    if hasattr(value, "tolist"):
        return value.tolist()
    return str(value)


VLM_EXECUTED = False
vlm_markdown = ""
vlm_pages = []
official_output_dir = OUTPUT_DIR / "paddleocr_vl_official"
official_output_dir.mkdir(exist_ok=True)

if AUTOMATED_CHECK:
    print("저장소 자동검사에서는 실제 모델 추론을 실행하지 않습니다.")
    print("Colab의 일반 실행에서는 이 분기를 사용하지 않습니다.")
else:
    from paddleocr import PaddleOCRVL

    vlm_pipeline = PaddleOCRVL(
        pipeline_version="v1.6",
        engine=VLM_ENGINE,
        device="gpu",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
    )
    page_results = list(vlm_pipeline.predict(str(VLM_INPUT_PATH)))
    if not page_results:
        raise RuntimeError("VLM이 페이지 결과를 반환하지 않았습니다.")

    for page_number, page_result in enumerate(page_results, start=1):
        page_result.save_to_json(save_path=official_output_dir)
        page_result.save_to_markdown(save_path=official_output_dir)

        raw_payload = getattr(page_result, "json", {})
        if callable(raw_payload):
            raw_payload = raw_payload()
        raw_payload = raw_payload.get("res", raw_payload)

        markdown_payload = getattr(page_result, "markdown", {})
        if callable(markdown_payload):
            markdown_payload = markdown_payload()
        markdown_text = (
            markdown_payload.get("markdown_texts", "")
            if isinstance(markdown_payload, dict)
            else str(markdown_payload or "")
        )
        if isinstance(markdown_text, list):
            markdown_text = "\n\n".join(map(str, markdown_text))

        if not markdown_text:
            blocks = raw_payload.get("parsing_res_list", [])
            markdown_text = "\n\n".join(
                str(block.get("block_content", ""))
                for block in blocks
                if block.get("block_content")
            )
        vlm_pages.append({
            "page": page_number,
            "markdown": markdown_text,
            "raw": raw_payload,
        })

    vlm_markdown = "\n\n".join(
        page["markdown"] for page in vlm_pages if page["markdown"]
    )
    if not vlm_markdown.strip():
        raise RuntimeError("모델은 실행됐지만 읽을 수 있는 문서 내용이 없습니다.")
    VLM_EXECUTED = True

vlm_run_record = {
    "model_executed": VLM_EXECUTED,
    "pipeline": VLM_PIPELINE_NAME,
    "vlm_model": VLM_MODEL_NAME,
    "engine": VLM_ENGINE,
    "input_file": INPUT_FILE_NAME,
    "pages": vlm_pages,
    "automated_repository_check": AUTOMATED_CHECK,
}
raw_path = OUTPUT_DIR / "paddleocr_vl_raw.json"
raw_path.write_text(
    json.dumps(
        vlm_run_record,
        ensure_ascii=False,
        indent=2,
        default=json_safe,
    ) + "\n",
    encoding="utf-8",
)
if VLM_EXECUTED:
    markdown_path = OUTPUT_DIR / "paddleocr_vl_result.md"
    markdown_path.write_text(vlm_markdown + "\n", encoding="utf-8")

print("모델 실행 완료:", VLM_EXECUTED)
if VLM_EXECUTED:
    print("\n--- 실제 VLM Markdown 앞부분 ---")
    print(vlm_markdown[:3000])
    print("\n저장:", raw_path, markdown_path)

complete_lab_step(5, 9, '`모델 실행 완료: True`와 실제 Markdown 일부를 확인합니다.')


In [ ]:
#@title 🟢 업무 JSON 변환과 비교 — 그대로 실행 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# `vlm_markdown`은 실제 모델 출력이고 `vlm_receipt`은 이를 업무 스키마로 옮긴 결과입니다. `model_executed`와 두
# 경로의 값 차이를 확인합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(6, 9, '업무 JSON 변환과 비교', '실제 VLM Markdown을 업무 JSON으로 바꾸고 OCR+규칙 기준선과 비교합니다.', '세 결과 파일과 모델 실행 여부·총액·원문 근거를 확인합니다.', '`vlm_markdown`은 실제 모델 출력이고 `vlm_receipt`은 이를 업무 스키마로 옮긴 결과입니다. `model_executed`와 두 경로의 값 차이를 확인합니다.', 'none')

ocr_receipt = extract_receipt_from_text(
    ocr_baseline_text,
    "OCR 원문 + Python 규칙",
)
ocr_receipt["provenance"] = {
    "model_executed": False,
    "engine": "course_rule_extractor",
    "input": OCR_BASELINE_SOURCE,
    "disclaimer": "VLM 결과가 아니라 비교용 OCR+규칙 기준선입니다.",
}

vlm_receipt = extract_receipt_from_text(
    vlm_markdown,
    "PaddleOCR-VL-1.6 실제 Markdown + Python 업무 규칙",
)
vlm_receipt["provenance"] = {
    "model_executed": VLM_EXECUTED,
    "pipeline": VLM_PIPELINE_NAME,
    "vlm_model": VLM_MODEL_NAME,
    "engine": VLM_ENGINE,
    "input_file": INPUT_FILE_NAME,
    "postprocess": "공개된 Python 규칙으로 업무 스키마 변환",
}

comparison = {
    field: {
        "ocr_rule": ocr_receipt.get(field),
        "actual_vlm": vlm_receipt.get(field),
        "must_check_source": True,
    }
    for field in ("store_name", "date", "total_amount", "items")
}
comparison_path = OUTPUT_DIR / "vlm_comparison.json"
comparison_path.write_text(
    json.dumps({
        "model_executed": VLM_EXECUTED,
        "model": VLM_MODEL_NAME,
        "input_file": INPUT_FILE_NAME,
        "comparison": comparison,
        "important": "두 결과 모두 원본 이미지와 사람이 대조해야 합니다.",
    }, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

assert ocr_receipt["total_amount"] is None or isinstance(
    ocr_receipt["total_amount"], int
)
if 실습_자료 == "제공 예제":
    assert ocr_receipt["total_amount"] == 76000
if not AUTOMATED_CHECK:
    assert VLM_EXECUTED, "실제 VLM 실행이 완료되지 않았습니다."

ocr_path = OUTPUT_DIR / "receipt.json"
vlm_path = OUTPUT_DIR / "receipt_vlm.json"
ocr_path.write_text(
    json.dumps(ocr_receipt, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
vlm_path.write_text(
    json.dumps(vlm_receipt, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(json.dumps({
    "실제 VLM 실행": VLM_EXECUTED,
    "VLM 모델": VLM_MODEL_NAME,
    "OCR·규칙 총액": ocr_receipt["total_amount"],
    "실제 VLM 총액": vlm_receipt["total_amount"],
    "실제 VLM 총액 근거": vlm_receipt["evidence"]["total_amount"],
}, ensure_ascii=False, indent=2))
print("✅ 실습 완료:", ocr_path, vlm_path, comparison_path)
download_artifact(ocr_path)
download_artifact(vlm_path)
download_artifact(comparison_path)

complete_lab_step(6, 9, '세 결과 파일과 모델 실행 여부·총액·원문 근거를 확인합니다.')


## 내가 직접 채우는 5줄

아래 셀에서 원본 대조가 가장 중요한 필드 하나와 처리 결정을
입력합니다. 막히면 바로 다음 정답 셀을 열어 비교합니다.


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `my_review`의 세 `None`만 채웁니다. 중요 필드, 원문 근거 유무, Excel 저장 전 행동을 직접 결정합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(7, 9, '내 검토 결정 입력', '중요 필드·근거 유무·저장 전 행동을 직접 정합니다.', '빈칸 안내 또는 내가 내린 검토 결정이 표시되어야 합니다.', '`my_review`의 세 `None`만 채웁니다. 중요 필드, 원문 근거 유무, Excel 저장 전 행동을 직접 결정합니다.', 'required')

# TODO: None 세 곳을 채우세요.
my_review = {
    "field": None,
    "evidence_found": None,
    "action": None,
}
if None in my_review.values():
    print("빈칸이 있습니다. 아래 힌트·정답 셀과 비교하세요.")
else:
    print("내 검토 결정:", my_review)

complete_lab_step(7, 9, '빈칸 안내 또는 내가 내린 검토 결정이 표시되어야 합니다.')


<details>
<summary>힌트와 전체 정답 보기</summary>

영향이 큰 `total_amount`를 선택하고, 원본 근거가 있으면
`Excel 저장 전 원본 검토`로 둡니다.
</details>


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `ANSWER_REVIEW`는 총액의 원문 근거가 있더라도 저장 전에 사람이 검토해야 한다는 공개 정답입니다. 수정 없이 실행합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(8, 9, '검토 정답 확인', '총액과 원문 근거를 기준으로 공개 검토 결정을 확인합니다.', '`Excel 저장 전 원본 검토`가 포함된 정답을 확인합니다.', '`ANSWER_REVIEW`는 총액의 원문 근거가 있더라도 저장 전에 사람이 검토해야 한다는 공개 정답입니다. 수정 없이 실행합니다.', 'none')

ANSWER_REVIEW = {
    "field": "total_amount",
    "evidence_found": bool(
        vlm_receipt["evidence"]["total_amount"]["raw_value"]
    ),
    "action": (
        "Excel 저장 전 원본 검토"
        if vlm_receipt["evidence"]["total_amount"]["raw_value"]
        else "사람이 원본부터 다시 확인"
    ),
}
assert ANSWER_REVIEW["action"] in {
    "Excel 저장 전 원본 검토",
    "사람이 원본부터 다시 확인",
}
print("전체 정답:", ANSWER_REVIEW)

complete_lab_step(8, 9, '`Excel 저장 전 원본 검토`가 포함된 정답을 확인합니다.')


## 선택 실험: 다른 자료로 한 번 더 확인하기

필수 실습을 먼저 끝낸 뒤, 인터넷에서 찾은 공개 문서나 개인정보를
가린 자료 한 장으로 같은 과정을 반복합니다. 결과가 잘 나오지 않아도
실패한 위치와 다음 질문을 남기면 실험이 완료됩니다.


In [ ]:
#@title 🔵 선택: 다른 자료 실험 기록 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# 이 셀은 선택 실험 기록지입니다. 위쪽 입력칸만 채우면 자료 출처, 잘된 점, 실패한 점, 다음 질문을 Markdown 파일로 저장합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(9, 9, '다른 자료 실험 기록', '인터넷에서 찾은 공개 자료나 비식별 자료의 결과를 네 줄로 정리합니다.', '`lesson04_research_note.md` 파일과 기록 내용이 표시되어야 합니다.', '이 셀은 선택 실험 기록지입니다. 위쪽 입력칸만 채우면 자료 출처, 잘된 점, 실패한 점, 다음 질문을 Markdown 파일로 저장합니다.', 'optional')

# RESEARCH_NOTE_CELL
# TODO(선택): 다른 자료로 다시 실험했다면 아래 입력칸만 채우세요.
자료_구분 = "제공 예제" #@param ["제공 예제", "공개 웹 자료", "비식별 개인 자료", "회사 승인 자료"]
자료_이름_또는_URL = "" #@param {type:"string"}
문서_종류 = "영수증" #@param ["영수증", "견적서", "신청서", "거래명세서", "표 캡처", "기타"]
잘된_점 = "" #@param {type:"string"}
실패한_점 = "" #@param {type:"string"}
다음_질문 = "" #@param {type:"string"}

research_focus = '추출된 필드와 빠진 필드, 원문 근거가 없는 값을 구분해 기록합니다.'
note = f'''# {문서_종류} 실험 기록

- 자료 구분: {자료_구분}
- 자료 이름 또는 원문 URL: {자료_이름_또는_URL or "미입력"}
- 이번 교시 관찰 질문: {research_focus}
- 잘된 점: {잘된_점 or "미입력"}
- 실패하거나 이상한 점: {실패한_점 or "미입력"}
- 다음에 바꿔 볼 한 가지: {다음_질문 or "미입력"}
'''
note_path = OUTPUT_DIR / "lesson04_research_note.md"
note_path.write_text(note + "\n", encoding="utf-8")
try:
    from IPython.display import Markdown, display
    display(Markdown(note))
except ImportError:
    print(note)
print("실험 기록 저장:", note_path)

complete_lab_step(9, 9, '`lesson04_research_note.md` 파일과 기록 내용이 표시되어야 합니다.')
